# equipo_proyecto_avance4

Comparación de una red totalmente conectada de referencia, la CNN original y la CNN con dos convoluciones adicionales. Diagnóstico de sobreajuste y regularización.

**Procedencia:** los experimentos se ejecutaron desde este repositorio con DirectML en una Radeon RX 6700S. La MLP es una referencia reconstruida y no el modelo histórico identificado del Avance 3. Las métricas corresponden a validación, no a un test independiente.

Las figuras se conservan en PNG y PDF; los históricos, matrices y predicciones se guardan en CSV. Para volver a entrenar, use una carpeta nueva de salida y active `RUN_TRAINING`.

In [ ]:
from pathlib import Path
import sys, json, inspect, subprocess
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/model.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "reports/avance4_2026-10-21"
RUN_TRAINING = False
print("Proyecto:", ROOT)
print("Evidencia de ejecuciones:", OUTPUT)

## 1. Datos y protocolo

Se usan 12,753 imágenes de entrenamiento y 3,712 de validación, 104 clases, resolución efectiva 112×112, lote 64 y semilla 0. Las etiquetas son enteras y los píxeles se normalizan a [0,1]. Solo entrenamiento usa aumentos. El mejor checkpoint se selecciona por F1 macro. Las dos CNN sin la intervención adicional usan Adam a 1e-3, dropout 0.5 y paciencia 5. La MLP usa Adam a 1e-4, paciencia 10 y mínimo 10 épocas, tras el colapso del primer intento a 1e-3. Ese ensayo se conserva en diagnósticos. La intervención combinada usa AdamW, reducción del LR y paciencia 12.

Una sola semilla no permite afirmar significancia estadística. La validación usada para selección no es un test final independiente.

In [ ]:
from src.config import load_config
cfg = load_config(OUTPUT / "cnn_original.yaml")
display({"data": cfg["data"], "train": cfg["train"], "runtime": cfg["runtime"]})
for key in ["cnn_original", "cnn_dos_capas_extra", "cnn_regularizada", "mlp_referencia"]:
    profile = load_config(OUTPUT / f"{key}.yaml")
    display({"experiment": key, "model": profile.get("model", {"architecture": "baseline"}), "train": profile["train"]})
manifest = json.loads((OUTPUT / "manifest.json").read_text(encoding="utf-8"))
print("Commit:", manifest["git_commit"])
print("Checkpoint elegido por:", manifest["selection_metric"])

## 2. Código y representaciones de los modelos

La versión ampliada agrega una convolución después de pool3 y otra después de pool4. Se muestran las implementaciones reales utilizadas para entrenar, no una arquitectura ilustrativa distinta.

In [ ]:
from src.train_directml import FlowerCNN, train_epoch, evaluate
from src.train_mlp_reference import FlowerMLP
print(inspect.getsource(FlowerCNN))
print(inspect.getsource(FlowerMLP))
display(Image(filename=str(OUTPUT / "figures/arquitecturas.png"), width=1100))

## 3. Entrenamiento reproducible

Estos comandos utilizan el mismo código que produjo los resultados guardados. Se usa una carpeta nueva para conservar las ejecuciones anteriores. Cambie `RUN_TRAINING=True` para ejecutarlos; una revisión normal del notebook carga los resultados existentes.

La CNN original y la ampliada se entrenan primero. Se añade la variante regularizada y se entrena la referencia totalmente conectada con el cargador y el ciclo existentes.

In [ ]:
NEW_OUTPUT = ROOT / "reports" / "avance4_repeticion"
if RUN_TRAINING:
    subprocess.run([sys.executable, "-u", "-m", "scripts.run_avance4", "--output", str(NEW_OUTPUT)], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "-u", "-m", "src.train_mlp_reference", "--config", str(NEW_OUTPUT / "cnn_original.yaml"), "--output", str(NEW_OUTPUT), "--lr", "0.0001", "--patience", "10", "--min-epochs", "10"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "-u", "-m", "scripts.prepare_avance4", "--output", str(NEW_OUTPUT)], cwd=ROOT, check=True)
    OUTPUT = NEW_OUTPUT
else:
    print("Se cargan las ejecuciones completas; RUN_TRAINING=False.")

In [ ]:
print("Ciclo de entrenamiento compartido:")
print(inspect.getsource(train_epoch))
print("Evaluación sobre imágenes sin aumentos:")
print(inspect.getsource(evaluate))

## 4. Resultados finales y comparación

La exactitud y F1 macro pertenecen al mismo checkpoint. Se compara explícitamente MLP frente a CNN y CNN original frente a CNN con dos capas adicionales. La convolución comparte filtros y utiliza vecindarios espaciales; la MLP recibe píxeles aplanados. El aumento de profundidad no garantiza una mejora de validación.

In [ ]:
results = pd.read_csv(OUTPUT / "resultados.csv")
display(results[["model", "parameters", "best_epoch", "epochs_run", "val_accuracy", "val_f1_macro", "val_loss", "train_seconds"]])
display(Image(filename=str(OUTPUT / "figures/comparacion_metricas.png"), width=1100))
indexed = results.set_index("experiment")
for reference, candidate in [("mlp_referencia", "cnn_original"), ("cnn_original", "cnn_dos_capas_extra"), ("cnn_original", "cnn_regularizada")]:
    print(reference, "->", candidate)
    print("Cambio de exactitud (puntos porcentuales):", round(100*(indexed.loc[candidate,"val_accuracy"]-indexed.loc[reference,"val_accuracy"]),2))
    print("Cambio de F1 macro:", round(indexed.loc[candidate,"val_f1_macro"]-indexed.loc[reference,"val_f1_macro"],4))

## 5. Diagnóstico de sobreajuste y regularización

Se comparan pérdida y exactitud de entrenamiento frente a validación. Una pérdida de entrenamiento que disminuye mientras validación se estanca o empeora es evidencia de sobreajuste. Entrenamiento se mide con aumentos y dropout, por lo que una diferencia puntual de exactitudes no basta para diagnosticarlo.

Dropout, aumentos y parada temprana están presentes en todos los modelos. La intervención adicional combina AdamW (weight decay 0.01, sin sesgos), reducción del LR por factor 0.5 con paciencia 2, mínimo 1e-5 y parada temprana con paciencia 12. El experimento mide el efecto combinado, no el aporte aislado de cada técnica.

In [ ]:
display(Image(filename=str(OUTPUT / "figures/comparacion_curvas.png"), width=1000))
display(results[["model", "train_accuracy_at_checkpoint", "val_accuracy", "accuracy_gap_at_checkpoint"]])
for row in results.itertuples():
    run_dir = next((OUTPUT / "runs" / row.experiment).glob("*/summary.json")).parent
    history = pd.read_csv(run_dir / "history.csv")
    best_loss = history.loc[history.val_loss.idxmin()]
    print(row.model, "| mínima pérdida val:", round(best_loss.val_loss,4), "en época", int(best_loss.epoch)+1,
          "| última pérdida train/val:", round(history.iloc[-1].loss,4), round(history.iloc[-1].val_loss,4))

## 6. Modelo elegido, matrices de confusión y errores

Se elige el mayor F1 macro de validación. La matriz normalizada muestra recall en la diagonal (filas reales, columnas predichas). Los ejemplos muestran seis aciertos y seis errores elegidos con semilla fija; esa selección ilustrativa no representa la frecuencia de errores del modelo.

In [ ]:
selected = results.sort_values("val_f1_macro", ascending=False).iloc[0]
key = selected.experiment
print("Modelo elegido:", selected.model, "| F1 macro:", selected.val_f1_macro, "| exactitud:", selected.val_accuracy)
run_dir = next((OUTPUT / "runs" / key).glob("*/summary.json")).parent
evaluation = json.loads((run_dir / "eval_report.json").read_text(encoding="utf-8"))
display(pd.DataFrame(evaluation["confused_pairs"]))
display(Image(filename=str(OUTPUT / "figures" / key / "confusion_matrix_normalized.png"), width=1100))
display(Image(filename=str(OUTPUT / "figures" / key / "predicciones_ejemplo.png"), width=1000))

## 7. Comprobación de las métricas y artefactos

Se recalculan exactitud y F1 desde las predicciones individuales, se comprueba que la matriz contiene 3,712 casos y se verifica que cada corrida tiene sus figuras. La evaluación también compara los logits del checkpoint de GPU y su exportación a Keras.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
for row in results.itertuples():
    run_dir = next((OUTPUT / "runs" / row.experiment).glob("*/summary.json")).parent
    pred = pd.read_csv(run_dir / "evaluation/validation_predictions.csv")
    cm = pd.read_csv(run_dir / "evaluation/confusion_matrix.csv", index_col=0)
    np.testing.assert_allclose(accuracy_score(pred.true_label, pred.predicted_label), row.val_accuracy, atol=1e-9)
    np.testing.assert_allclose(f1_score(pred.true_label, pred.predicted_label, average="macro"), row.val_f1_macro, atol=0.000051)
    assert cm.to_numpy().sum() == len(pred) == 3712
    assert cm.shape == (104,104)
    for name in ["curvas_entrenamiento_validacion", "confusion_matrix", "confusion_matrix_normalized", "predicciones_ejemplo"]:
        assert (OUTPUT / "figures" / row.experiment / f"{name}.png").exists()
    print(row.model, "OK:", len(pred), "predicciones y matriz 104×104")

## 8. Conclusiones y entrega

La tabla y las curvas sustentan la elección por F1 macro. Las métricas corresponden a una semilla y al conjunto usado para seleccionar los modelos. No se atribuye a Avance 3 la referencia MLP reconstruida, ni a DM los resultados de entrenamiento con todas las imágenes reales.

Se corrigieron resolución, lote, descripción de arquitecturas, separación entre destilación y clasificación, selección del checkpoint y alcance de validación. Se incorporaron las representaciones solicitadas. No se afirma atender comentarios de retroalimentación cuyo texto no fue proporcionado.

El PDF actualizado contiene las secciones 1–8. Los archivos de entrega se llaman `equipo_proyecto_avance4.pdf` y `equipo_proyecto_avance4.ipynb`. Las figuras originales se conservan para reutilizarlas en la presentación.